In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [ ]:
train_dataset = ImageFolder(
    root="data/train",
    transform=train_transform,
)

evaluation_dataset = ImageFolder(
    root="data/test",
    transform=evaluation_transform,
)

In [ ]:
print("Train:", len(train_dataset))
print("Evaluation:", len(evaluation_dataset))

print("Classes:", train_dataset.class_to_idx)

Train: 800
Evaluation: 201
Classes: {'curly': 0, 'straight': 1}


In [ ]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [ ]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("First 10 labels:", labels[:10])

Images shape: torch.Size([20, 3, 200, 200])
Labels shape: torch.Size([20])
First 10 labels: tensor([1, 1, 1, 0, 0, 1, 0, 1, 1, 0])


In [ ]:
import torch.nn as nn

In [ ]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=0,
        )

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.hidden = nn.Linear(32 * 99 * 99, 64)

        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))

        x = x.flatten(start_dim=1)

        x = nn.functional.relu(self.hidden(x))

        return self.output(x)

In [ ]:
model = HairModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

In [ ]:
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Total parameters:", total_params)

Total parameters: 20073473


In [ ]:
train_loss = []
train_accuracy = []

evaluation_loss = []
evaluation_accuracy = []

In [ ]:
model.train()

HairModel(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (hidden): Linear(in_features=313632, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)

In [ ]:
for epoch in range(10):
    print(f"Epoch {epoch + 1}/10")
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        logits = model(images)
        torch.Size([20, 1])
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 10/10


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

data_path = '/content/drive/MyDrive/zoomcamp_homework/hw8/data'

os.listdir(data_path)

['test', 'train']

In [6]:
import os

os.listdir('/content/drive/MyDrive/zoomcamp_homework/hw8/data')

['test', 'train']